# 01c Cleaning and airport network
We clean the data and build the airport edge table for RQ2. Only flights with an impossible scheduled duration are removed. Cancelled and diverted flights stay in the data, because RQ2 counts every scheduled flight between two airports; later notebooks drop them where needed.

**Before running:** select the "Python (ctds)" kernel and run the notebook from the notebooks/ folder (see the README).

In [1]:
# imports
import sys
sys.path.append("..")  # lets us import the src/ folder from the project root

import pandas as pd
from IPython.display import display

from src.config import CLEAN_PARQUET
from src.cleaning import make_clean_parquet
from src.data_loading import load_flights
from src.network import make_edges_parquet


# settings
pd.set_option("display.float_format", "{:,.4f}".format)

## 1. Cleaning
We remove a flight when its scheduled duration is missing or not positive, when its implied speed (distance divided by scheduled duration) is above 650 mph, well above an airliner's cruising speed, or when its scheduled duration is under half or over twice the median for its route (origin, dest). Route medians come from January–June; a route that first appears later uses its own median. The rule looks only at scheduled times and distances, never at delays. Cancelled and diverted flights stay in, and a new status column says which is which. A scheduled time of 2400 becomes 0.

The three rules flag 2, 5 and 38 flights. Some flights break more than one rule, so 41 of the 7,079,081 flights are removed and 7,079,040 remain.

In [2]:
# clean the full data and save flights_2024_clean.parquet
# the report counts how many flights each rule flags
report = make_clean_parquet()
display(report)


# read back only the columns we want to check
clean = load_flights(CLEAN_PARQUET, columns=["status", "crs_dep_time", "crs_arr_time"])
status_counts = clean["status"].value_counts()

print(f"Clean flights: {len(clean):,}")
display(status_counts.to_frame())


# 2400 is now 0, so no scheduled time should be above 2359
max_dep_time = clean["crs_dep_time"].max()
max_arr_time = clean["crs_arr_time"].max()
print("Max scheduled dep/arr time:", max_dep_time, max_arr_time)
assert max_dep_time <= 2359 and max_arr_time <= 2359, "a scheduled time is still above 2359"

[clean_flights] runtime: 46.72 s | peak memory (Python/pandas): 1344.4 MB | peak memory (pyarrow pool): 1035.6 MB


,flights_flagged,share_pct
duration missing or <= 0,2,0.0000
speed > 650 mph,5,0.0001
duration < 0.5x or > 2x route median,38,0.0005
total removed (any rule),41,0.0006


Clean flights: 7,079,040

,count
status,
completed,6965246
cancelled,96309
diverted,17485


Max scheduled dep/arr time: 2359 2359


## 2. Airport edges (RQ2)
This table is the input for RQ2: one row per direction (origin → dest) with the number of scheduled flights, cancelled and diverted included. It uses January–June only, so RQ2 never sees the months RQ3 validates and tests on. The column distance = 1 / flights is not the flight distance in miles; it is the edge length for betweenness, so busy routes count as short. There are 6,545 directed edges between 343 airports, covering 3,461,313 flights. The other 5 of the 348 airports only have flights after June.

In [3]:
# build the edges from months 1-6 and save them for notebook 03
edges = make_edges_parquet()


# airports at either end of an edge
airport_codes = edges[["origin", "dest"]].astype(str).values.ravel()
airports = pd.unique(airport_codes)
total_flights = edges["flights"].sum()

print(f"Edges: {len(edges):,} | airports: {len(airports)} | flights: {total_flights:,}")


# busiest routes
busiest = edges.sort_values("flights", ascending=False)
display(busiest.head(10))

[build_edges] runtime: 0.37 s | peak memory (Python/pandas): 171.8 MB | peak memory (pyarrow pool): 1028.8 MB
Edges: 6,545 | airports: 343 | flights: 3,461,313


,origin,dest,flights,distance
3876,OGG,HNL,5829,0.0002
3229,HNL,OGG,5819,0.0002
3326,LAX,SFO,5482,0.0002
4192,SFO,LAX,5460,0.0002
410,LGA,ORD,5366,0.0002
3477,ORD,LGA,5365,0.0002
3117,PHX,DEN,4846,0.0002
4490,DEN,PHX,4810,0.0002
2484,BOS,DCA,4746,0.0002
3436,DCA,BOS,4746,0.0002
